In [0]:
#Build the game attribute master
ap=r'C:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\game_attr_cleaned.csv'
sp=r'C:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\sales_clean.csv'
mp=r'C:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\mapping_clean.csv'
bp=r"C:\EveriTools\Workspace\Code\DemandForeCast\Data\raw sample\IGT_Everi_Attributes.xlsx"
ep=r"C:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\performance_aggregated.csv"

In [0]:
import pandas as pd

ap_df = pd.read_csv(ap)
sp_df = pd.read_csv(sp)
mp_df = pd.read_csv(mp)
bp_df = pd.read_excel(bp)
ep_df = pd.read_csv(ep)

print('game_attr_cleaned.csv head:')
display(ap_df.head())

print('sales_clean.csv head:')
display(sp_df.head())

In [0]:
ep_df.head()

In [0]:
target_companies = {
    'igt canada',
    'igt canada solutions ulc',
    'igt n.a.'
}

sp_df_filtered = sp_df[
    sp_df['company_code_desc'].str.strip().str.lower().isin(target_companies)
    & (sp_df['co_pa_account_assign_desc'].str.strip().str.lower() == 'new machine')
]

display(sp_df_filtered)
print(f"Filtered rows: {len(sp_df_filtered)}")

In [0]:
ap_df_lower = ap_df.copy()

# Lowercase all column names
ap_df_lower.columns = ap_df_lower.columns.str.strip().str.lower()

# Lowercase all string values
for col in ap_df_lower.select_dtypes(include='object').columns:
    ap_df_lower[col] = ap_df_lower[col].str.strip().str.lower()

ap_df_filtered = ap_df_lower[
    ap_df_lower['source_company_cd'] == 'igt'
]

display(ap_df_filtered)
print(f"Filtered rows: {len(ap_df_filtered)}")

In [0]:
ap_df

In [0]:
# Join ap_df(theme_nk) to sp_df(theme_key)
ap_join = ap_df.copy()
sp_join = sp_df.copy()

ap_join['theme_nk_join'] = ap_join['theme_nk'].astype(str).str.strip().str.lower()
sp_join['theme_key_join'] = sp_join['theme_key'].astype(str).str.strip().str.lower()

joined_df = ap_join.merge(
    sp_join,
    left_on='theme_nk_join',
    right_on='theme_key_join',
    how='inner',
    suffixes=('_ap', '_sp')
)

display(joined_df.head())
print(f"Joined rows: {len(joined_df)}")

In [0]:
# Join ap_df(theme_nk) to mp_df(theme_key)
ap_join_mp = ap_df.copy()
mp_join = mp_df.copy()

ap_join_mp['theme_nk_join'] = ap_join_mp['theme_nk'].astype(str).str.strip().str.lower()
mp_join['theme_key_join'] = mp_join['theme_key'].astype(str).str.strip().str.lower()

joined_df_mp = ap_join_mp.merge(
    mp_join,
    left_on='theme_nk_join',
    right_on='theme_key_join',
    how='inner',
    suffixes=('_ap', '_mp')
)

display(joined_df_mp.head())
print(f"Joined rows: {len(joined_df_mp)}")

In [0]:
joined_df_mp

In [0]:
# Coverage of Everi game themes between ep_df and bp_df using game-name join

# Normalize key columns for robust joining
ep_work = ep_df.copy()
bp_work = bp_df.copy()

# Helper for case-insensitive column lookup
def _find_col(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        key = c.strip().lower()
        if key in lookup:
            return lookup[key]
    return None

ep_supplier_col = _find_col(ep_work, ['supplier'])
ep_game_col = _find_col(ep_work, ['game_name', 'game name'])
# EP has no explicit themename, so default to game name as theme identifier
ep_theme_col = _find_col(ep_work, ['themename', 'theme_name', 'theme name']) or ep_game_col
# BP uses Theme Name in this file; keep Game Name as first preference if present
bp_game_col = _find_col(bp_work, ['Game Name', 'game_name', 'game name', 'Theme Name', 'theme name'])

if not ep_supplier_col or not ep_game_col or not bp_game_col:
    print('Could not find required columns.')
    print('ep_df columns:', list(ep_work.columns))
    print('bp_df columns:', list(bp_work.columns))
    raise KeyError('Missing one or more required columns for analysis.')

ep_work['supplier_norm'] = ep_work[ep_supplier_col].astype(str).str.strip().str.lower()
ep_work['game_name_join'] = ep_work[ep_game_col].astype(str).str.strip().str.lower()
bp_work['game_name_join'] = bp_work[bp_game_col].astype(str).str.strip().str.lower()

# Restrict EP population to IGT/Everi, then isolate Everi for requested percentages
ep_igt_everi = ep_work[ep_work['supplier_norm'].isin(['igt', 'everi'])].copy()
ep_everi = ep_igt_everi[ep_igt_everi['supplier_norm'] == 'everi'].copy()

# Outer join gives left/right/inner populations via indicator
merged = ep_everi.merge(
    bp_work,
    on='game_name_join',
    how='outer',
    indicator=True,
    suffixes=('_ep', '_bp')
)

# Unique theme populations from EP side (theme identifier = ep_theme_col)
left_themes = set(merged.loc[merged['_merge'] == 'left_only', ep_theme_col].dropna().astype(str).str.strip())
inner_themes = set(merged.loc[merged['_merge'] == 'both', ep_theme_col].dropna().astype(str).str.strip())

everi_theme_universe = set(ep_everi[ep_theme_col].dropna().astype(str).str.strip())
right_unique_bp_games = merged.loc[merged['_merge'] == 'right_only', bp_game_col].dropna().astype(str).str.strip().nunique()

theme_total = len(everi_theme_universe)
unmatched_pct = (len(left_themes) / theme_total * 100) if theme_total else 0
matched_pct = (len(inner_themes) / theme_total * 100) if theme_total else 0
right_pct_vs_everi_theme_universe = (right_unique_bp_games / theme_total * 100) if theme_total else 0

summary = pd.DataFrame({
    'population': ['left_only (EP Everi unmatched)', 'inner (matched in both)', 'right_only (BP-only names)'],
    'count_unique': [len(left_themes), len(inner_themes), right_unique_bp_games],
    'pct_of_ep_everi_unique_theme_universe': [unmatched_pct, matched_pct, right_pct_vs_everi_theme_universe]
})

print('Using columns ->')
print(f"EP supplier: {ep_supplier_col} | EP game: {ep_game_col} | EP theme-id: {ep_theme_col} | BP game/theme: {bp_game_col}")
print('EP filtered suppliers:', sorted(ep_igt_everi['supplier_norm'].dropna().unique().tolist()))
print('EP Everi rows:', len(ep_everi))
print('EP Everi unique theme universe:', theme_total)
print()
print('Requested percentage: Everi unique themes NOT included in both (left_only)')
print(f"{len(left_themes)} / {theme_total} = {unmatched_pct:.2f}%")
print()
display(summary)

In [0]:
# Join stats for both IGT and Everi against BP

def supplier_join_stats(ep_source, bp_source, suppliers=('igt', 'everi')):
    ep_local = ep_source.copy()
    bp_local = bp_source.copy()

    def _find_col(df, candidates):
        lookup = {str(c).strip().lower(): c for c in df.columns}
        for c in candidates:
            key = c.strip().lower()
            if key in lookup:
                return lookup[key]
        return None

    ep_supplier_col = _find_col(ep_local, ['supplier'])
    ep_game_col = _find_col(ep_local, ['game_name', 'game name'])
    ep_theme_col = _find_col(ep_local, ['themename', 'theme_name', 'theme name']) or ep_game_col
    bp_game_col = _find_col(bp_local, ['Game Name', 'game_name', 'game name', 'Theme Name', 'theme name'])

    if not ep_supplier_col or not ep_game_col or not bp_game_col:
        raise KeyError('Missing required columns in EP/BP dataframes')

    ep_local['supplier_norm'] = ep_local[ep_supplier_col].astype(str).str.strip().str.lower()
    ep_local['game_name_join'] = ep_local[ep_game_col].astype(str).str.strip().str.lower()
    bp_local['game_name_join'] = bp_local[bp_game_col].astype(str).str.strip().str.lower()

    out = []
    for supplier in suppliers:
        ep_supplier_df = ep_local[ep_local['supplier_norm'] == supplier].copy()

        merged_supplier = ep_supplier_df.merge(
            bp_local,
            on='game_name_join',
            how='outer',
            indicator=True,
            suffixes=('_ep', '_bp')
        )

        left_themes = set(merged_supplier.loc[merged_supplier['_merge'] == 'left_only', ep_theme_col].dropna().astype(str).str.strip())
        inner_themes = set(merged_supplier.loc[merged_supplier['_merge'] == 'both', ep_theme_col].dropna().astype(str).str.strip())
        right_unique_bp_games = merged_supplier.loc[merged_supplier['_merge'] == 'right_only', bp_game_col].dropna().astype(str).str.strip().nunique()

        supplier_theme_universe = set(ep_supplier_df[ep_theme_col].dropna().astype(str).str.strip())
        theme_total = len(supplier_theme_universe)

        out.append({
            'supplier': supplier,
            'ep_rows': len(ep_supplier_df),
            'ep_unique_theme_universe': theme_total,
            'left_only_unique': len(left_themes),
            'left_only_pct': (len(left_themes) / theme_total * 100) if theme_total else 0,
            'inner_unique': len(inner_themes),
            'inner_pct': (len(inner_themes) / theme_total * 100) if theme_total else 0,
            'right_only_unique_bp_names': right_unique_bp_games,
            'right_only_pct_vs_ep_universe': (right_unique_bp_games / theme_total * 100) if theme_total else 0
        })

    return pd.DataFrame(out), ep_game_col, ep_theme_col, bp_game_col

stats_df, ep_game_col_used, ep_theme_col_used, bp_game_col_used = supplier_join_stats(ep_df, bp_df, suppliers=('igt', 'everi'))

print(f"Join key used: EP [{ep_game_col_used}] -> BP [{bp_game_col_used}]")
print(f"Theme identifier used on EP side: {ep_theme_col_used}")
print()
display(stats_df)


In [0]:
from pathlib import Path

# Export unique theme names from EP/BP filtered to IGT/Everi

def _find_col(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        key = c.strip().lower()
        if key in lookup:
            return lookup[key]
    return None

# Resolve key columns
ep_supplier_col = _find_col(ep_df, ['supplier'])
ep_theme_col = _find_col(ep_df, ['themename', 'theme_name', 'theme name', 'game_name', 'game name'])
ep_release_col = _find_col(ep_df, ['game_release_date', 'game release date', 'release_date', 'release date'])

bp_vendor_col = _find_col(bp_df, ['vendor', 'supplier'])
bp_theme_col = _find_col(bp_df, ['Theme Name', 'theme_name', 'theme name', 'Game Name', 'game_name', 'game name'])

if not ep_supplier_col or not ep_theme_col or not bp_theme_col:
    raise KeyError('Required EP/BP columns not found for unique-theme export.')

# Filter to IGT + Everi
ep_filtered = ep_df[ep_df[ep_supplier_col].astype(str).str.strip().str.lower().isin(['igt', 'everi'])].copy()

if bp_vendor_col:
    bp_filtered = bp_df[bp_df[bp_vendor_col].astype(str).str.strip().str.lower().isin(['igt', 'everi'])].copy()
else:
    bp_filtered = bp_df.copy()

# Build EP unique theme-name output with min game release date
ep_filtered['_theme_name_clean'] = ep_filtered[ep_theme_col].astype(str).str.strip()
ep_filtered['_theme_name_clean'] = ep_filtered['_theme_name_clean'].replace('', pd.NA)

if ep_release_col:
    ep_filtered['_release_dt'] = pd.to_datetime(ep_filtered[ep_release_col], errors='coerce')
    ep_unique = (
        ep_filtered.dropna(subset=['_theme_name_clean'])
        .groupby('_theme_name_clean', as_index=False)['_release_dt']
        .min()
        .rename(columns={
            '_theme_name_clean': 'theme_name',
            '_release_dt': 'min_game_release_date'
        })
        .sort_values('theme_name', kind='stable')
        .reset_index(drop=True)
    )
    ep_unique['min_game_release_date'] = ep_unique['min_game_release_date'].dt.strftime('%Y-%m-%d')
else:
    ep_unique = pd.DataFrame({
        'theme_name': sorted(ep_filtered['_theme_name_clean'].dropna().unique())
    })
    ep_unique['min_game_release_date'] = pd.NA

# Build BP unique theme-name output
bp_unique = pd.DataFrame({
    'theme_name': sorted(bp_filtered[bp_theme_col].dropna().astype(str).str.strip().replace('', pd.NA).dropna().unique())
})

# Save in raw data folder
raw_data_dir = Path(r'C:\EveriTools\Workspace\Code\DemandForeCast\Data\raw sample')
raw_data_dir.mkdir(parents=True, exist_ok=True)

ep_out = raw_data_dir / 'ep_filtered_unique_theme_names.csv'
bp_out = raw_data_dir / 'bp_filtered_unique_theme_names.csv'

ep_unique.to_csv(ep_out, index=False)
bp_unique.to_csv(bp_out, index=False)

print(f'EP filtered unique themes: {len(ep_unique)} -> {ep_out}')
print(f'BP filtered unique themes: {len(bp_unique)} -> {bp_out}')
if ep_release_col:
    print(f'Min release date column used: {ep_release_col}')
else:
    print('No EP release-date column found; min_game_release_date left blank.')

display(ep_unique.head())
display(bp_unique.head())

In [0]:
ep_unique